In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml


# 26_580_643 rows
with open("configs.yaml", "r") as f:
    configs = yaml.safe_load(f)

In [3]:

iid = 0
SAVE_OUT_PATH = configs["PATH_TO_DATA_DIR"]
ALL_COLLECTIONS = configs["ALL_COLLECTIONS"]
path_to_data = f"{SAVE_OUT_PATH}/nuGun_pT_0_50_reco_{iid}.h5"
df_head = pd.read_hdf(path_to_data, key="df", stop=10)
print(df_head)


ImportError: Missing optional dependency 'pytables'.  Use pip or conda to install pytables.

In [ ]:

# Count unique cell IDs and hits per cell ID for inside_bounds hits, per subcollection
from collections import defaultdict

cell_hit_counts = {col: defaultdict(int) for col in ALL_COLLECTIONS}

for chunk in pd.read_hdf(path_to_data, key="df", chunksize=1_000_000):
    for col in ALL_COLLECTIONS:
        mask = (chunk["collection"] == col) & (chunk["inside_bounds"] == True)
        sub = chunk.loc[mask, "cellid0"]
        for cell_id, count in sub.value_counts().items():
            cell_hit_counts[col][cell_id] += count

for col in ALL_COLLECTIONS:
    counts = cell_hit_counts[col]
    if counts:
        n_unique = len(counts)
        hits_per_cell = list(counts.values())
        print(f"{col}: {n_unique} unique cell IDs, hits per cell — min={min(hits_per_cell)}, max={max(hits_per_cell)}, mean={sum(hits_per_cell)/n_unique:.2f}")
    else:
        print(f"{col}: 0 unique cell IDs")


In [ ]:

OUTPUT_COLS = ["Edep", "x", "y", "z", "t", "system", "side", "layer", "module", "sensor"]

results = {col: [] for col in ALL_COLLECTIONS}
counts_all = {col: 0 for col in ALL_COLLECTIONS}

for chunk in pd.read_hdf(path_to_data, key="df", chunksize=1_000_000):
    for col in ALL_COLLECTIONS:
        mask_all = chunk["collection"] == col
        counts_all[col] += mask_all.sum()
        mask = mask_all & (chunk["inside_bounds"] == True)
        results[col].append(chunk.loc[mask, OUTPUT_COLS])

for col in ALL_COLLECTIONS:
    print(f"length of {col}, all: {counts_all[col]}")
    arr = pd.concat(results[col]).to_numpy() if results[col] else np.empty((0, len(OUTPUT_COLS)))
    print(f"length of {col}, inside_bounds: {len(arr)}")
    np.save(f"{SAVE_OUT_PATH}/{col}_SimTrackerHit_conditional_reco_{iid}.npy", arr)

In [1]:
import sys
print(sys.executable)

/software/python-anaconda-2022.05-el8-x86_64/bin/python
